# Phase 03H: Teacher B Three-Seed Aggregate & Teacher C Entry

Notebook này **không huấn luyện** và **không mở Held-out Test**. Mục tiêu là:

1. xác minh ba run Phase 03G của Teacher B ở seed `42`, `123`, `2026`;
2. kiểm tra toàn bộ manifest/checksum, best/last checkpoint và quality gate;
3. tổng hợp mean, sample SD, CV và mức vượt threshold;
4. phát hành checkpoint registry chính thức cho KD;
5. chỉ sau khi tất cả gate đạt mới cho phép tạo Phase 03I Teacher C preflight.

## Kaggle inputs bắt buộc

- output notebook Phase 03G seed 42;
- output notebook Phase 03G seed 123;
- output notebook Phase 03G seed 2026.

Có thể gắn ba output riêng hoặc một dataset/notebook output chứa đủ ba thư mục run.
Internet nên để **Off**. GPU không cần thiết cho notebook aggregate này.


## Cell 1 — Environment, helper và discovery

### Công việc

- tạo output tree của Phase 03H;
- tìm đúng ba `TEACHER_B_SEED*_FULL_RUN_COMPLETE.json`;
- xác nhận seed `{42,123,2026}`;
- chưa đọc ảnh, caption, Test Parquet hoặc checkpoint payload.

### Đầu ra dự kiến

```text
PHASE 03H CELL 1 STATUS
Status                    : PASSED
Teacher B COMPLETE runs   : 3 / 3
Seeds                     : [42, 123, 2026]
Test Set status           : SEALED
Next                      : verify manifests and checkpoints
```


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase3_teacher_b_three_seed_aggregate"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
MODEL_CARD_DIR = OUTPUT_ROOT / "model_cards"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    RESULT_DIR,
    MODEL_CARD_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


teacher_markers = sorted(
    INPUT_ROOT.rglob("TEACHER_B_SEED*_FULL_RUN_COMPLETE.json")
)
assert len(teacher_markers) == 3, (
    "Cần gắn đúng ba output Phase 03G; tìm thấy "
    f"{len(teacher_markers)} marker: {teacher_markers}"
)
marker_headers = [read_json(path) for path in teacher_markers]
assert {int(item["seed"]) for item in marker_headers} == {42, 123, 2026}
assert all(item.get("teacher_id") == "B" for item in marker_headers)
assert all(item.get("test_set_status") == "sealed" for item in marker_headers)
assert not any(bool(item.get("test_set_used")) for item in marker_headers)

cell_audit = {
    "audit_version": "phase3h_cell01_discovery_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "python": platform.python_version(),
    "markers": [str(path) for path in teacher_markers],
    "seeds": sorted(int(item["seed"]) for item in marker_headers),
    "dataset_read": False,
    "caption_read": False,
    "checkpoint_payload_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(AUDIT_DIR / "phase3h_cell01_discovery.json", cell_audit)

print("=" * 72)
print("PHASE 03H CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher B COMPLETE runs   : 3 / 3")
print("Seeds                     : [42, 123, 2026]")
print("Dataset read              : False")
print("Caption read              : False")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Output root               :", OUTPUT_ROOT)
print("Next                      : verify manifests and checkpoints")


PHASE 03H CELL 1 STATUS
Status                    : PASSED
Teacher B COMPLETE runs   : 3 / 3
Seeds                     : [42, 123, 2026]
Dataset read              : False
Caption read              : False
Checkpoint payload read   : False
Test Set status           : SEALED
Test Set used             : False
Output root               : /kaggle/working//phase3_teacher_b_three_seed_aggregate
Next                      : verify manifests and checkpoints


## Cell 2 — Manifest, SHA256SUMS và checkpoint integrity

### Công việc

- xác minh marker hoàn tất, model/revision và quality gate;
- băm lại mọi artifact trong manifest của từng seed;
- kiểm tra best/last checkpoint theo hash đã công bố;
- đối chiếu `SHA256SUMS.txt`;
- **không** gọi `torch.load`, vì đây là kiểm tra byte-level.

### Thời gian dự kiến

Khoảng 5–15 phút do ba `last.pt` có dung lượng lớn.

### Đầu ra dự kiến

```text
Manifest rows verified    : 68
Best/last checkpoints     : 6 / 6 VERIFIED
Quality gates             : 3 / 3 PASSED
Checkpoint payload read   : False
```


In [2]:
CELL_STARTED_AT = time.perf_counter()
EXPECTED_MODEL = "Salesforce/blip-image-captioning-base"
EXPECTED_REVISION = "82a37760796d32b1411fe092ab5d4e227313294b"
run_records = []
manifest_rows_total = 0
sha_rows_total = 0
checkpoint_count = 0

for marker_path in teacher_markers:
    marker = read_json(marker_path)
    seed = int(marker["seed"])
    run_root = marker_path.resolve().parent

    assert marker["status"] == "complete"
    assert marker["teacher_id"] == "B"
    assert marker["model"] == EXPECTED_MODEL
    assert marker["revision"] == EXPECTED_REVISION
    assert int(marker["completed_epochs"]) == 10
    assert int(marker["optimizer_steps"]) == 35410
    assert marker["selection_split"] == "validation"
    assert marker["quality_gate_passed"] is True
    assert marker["qualified_for_kd"] is True
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    manifest_csv = resolve_inside(
        run_root, marker["manifests"]["csv"]["relative_path"]
    )
    manifest_json = resolve_inside(
        run_root, marker["manifests"]["json"]["relative_path"]
    )
    sha_file = resolve_inside(
        run_root, marker["manifests"]["sha256sums"]["relative_path"]
    )
    for key, path in [
        ("csv", manifest_csv),
        ("json", manifest_json),
        ("sha256sums", sha_file),
    ]:
        assert path.is_file(), path
        assert sha256_file(path) == marker["manifests"][key]["sha256"]

    with manifest_csv.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    manifest_json_payload = read_json(manifest_json)
    assert int(manifest_json_payload["artifact_count"]) == len(rows)
    assert len(manifest_json_payload["artifacts"]) == len(rows)

    expected_sha_lines = []
    for row in rows:
        artifact = resolve_inside(run_root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        actual_sha = sha256_file(artifact)
        assert actual_sha == row["sha256"], artifact
        expected_sha_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual_sha_lines = [
        line.strip()
        for line in sha_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert actual_sha_lines == expected_sha_lines

    for checkpoint_kind in ["best_checkpoint", "last_checkpoint"]:
        checkpoint_meta = marker[checkpoint_kind]
        checkpoint_path = resolve_inside(
            run_root, checkpoint_meta["relative_path"]
        )
        assert checkpoint_path.is_file()
        assert sha256_file(checkpoint_path) == checkpoint_meta["sha256"]
        checkpoint_count += 1

    manifest_rows_total += len(rows)
    sha_rows_total += len(actual_sha_lines)
    run_records.append(
        {
            "seed": seed,
            "run_root": str(run_root),
            "marker_path": str(marker_path.resolve()),
            "marker_sha256": sha256_file(marker_path),
            "marker": marker,
            "manifest_rows": len(rows),
            "manifest_csv_sha256": sha256_file(manifest_csv),
            "manifest_json_sha256": sha256_file(manifest_json),
            "sha256sums_sha256": sha256_file(sha_file),
        }
    )

assert sorted(item["seed"] for item in run_records) == [42, 123, 2026]
assert manifest_rows_total == 68, manifest_rows_total
assert sha_rows_total == 68, sha_rows_total
assert checkpoint_count == 6

write_json(
    AUDIT_DIR / "phase3h_cell02_integrity.json",
    {
        "audit_version": "phase3h_cell02_integrity_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "manifest_rows_verified": manifest_rows_total,
        "sha256sum_rows_verified": sha_rows_total,
        "best_last_checkpoints_verified": checkpoint_count,
        "quality_gates_passed": 3,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03H CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Manifest rows verified    :", manifest_rows_total)
print("SHA256SUMS rows verified  :", sha_rows_total)
print("Best/last checkpoints     : 6 / 6 VERIFIED")
print("Quality gates             : 3 / 3 PASSED")
print("Qualified for KD          : 3 / 3")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : audit metrics and protocol")


PHASE 03H CELL 2 STATUS
Status                    : PASSED
Manifest rows verified    : 68
SHA256SUMS rows verified  : 68
Best/last checkpoints     : 6 / 6 VERIFIED
Quality gates             : 3 / 3 PASSED
Qualified for KD          : 3 / 3
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : audit metrics and protocol


## Cell 3 — Metric, protocol và leakage audit

### Công việc

- xác minh metric hữu hạn và đủ 5.000 prediction mỗi seed;
- kiểm tra `quality_margin = CIDEr-D − threshold`;
- ghi nhận overflow retry;
- khóa compatibility: sequence KD được phép, feature KD cần projection,
  direct logit KD bị cấm vì vocabulary BLIP khác Student.

### Đầu ra dự kiến

```text
Finite metric runs        : 3 / 3 PASSED
Prediction counts         : 5000 / seed
Overflow retries          : {42: 10, 123: 8, 2026: 8}
Direct logit KD           : PROHIBITED
```


In [3]:
CELL_STARTED_AT = time.perf_counter()
metric_names = [
    "BLEU-1",
    "BLEU-4",
    "METEOR",
    "ROUGE-L",
    "CIDEr-D",
    "validation_ce",
]
overflow_by_seed = {}
for run in run_records:
    marker = run["marker"]
    best = marker["best_result"]
    assert int(best["epoch"]) in range(1, 11)
    assert int(best["prediction_count"]) == 5000
    assert best["test_set_status"] == "sealed"
    for metric_name in metric_names:
        assert math.isfinite(float(best[metric_name])), (
            run["seed"], metric_name, best[metric_name]
        )
    expected_margin = (
        float(best["CIDEr-D"]) - float(marker["quality_threshold"])
    )
    assert math.isclose(
        expected_margin,
        float(marker["quality_margin"]),
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    assert expected_margin >= 0.001
    overflow_by_seed[run["seed"]] = int(marker["overflow_retries"])

assert overflow_by_seed == {42: 10, 123: 8, 2026: 8}

KD_COMPATIBILITY = {
    "sequence_kd": {
        "allowed": True,
        "reason": "Teacher caption can supervise Student sequence.",
    },
    "feature_kd": {
        "allowed": True,
        "condition": (
            "A learned projection and explicit tensor-shape contract "
            "must be audited before training."
        ),
    },
    "direct_logit_kd": {
        "allowed": False,
        "reason": (
            "BLIP native tokenizer/vocabulary and Student shared "
            "vocabulary are not index-compatible."
        ),
    },
}
write_json(
    CONFIG_DIR / "teacher_b_kd_compatibility_contract.json",
    {
        "contract_version": "teacher_b_kd_compatibility_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "B",
        "model": EXPECTED_MODEL,
        "compatibility": KD_COMPATIBILITY,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase3h_cell03_metric_protocol.json",
    {
        "audit_version": "phase3h_cell03_metric_protocol_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "finite_metric_runs": 3,
        "prediction_count_per_seed": 5000,
        "overflow_retries": overflow_by_seed,
        "quality_margin_verified": True,
        "kd_compatibility": KD_COMPATIBILITY,
        "caption_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03H CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Finite metric runs        : 3 / 3 PASSED")
print("Prediction counts         : 5000 / seed")
print("Quality margin            : 3 / 3 VERIFIED")
print("Overflow retries          :", overflow_by_seed)
print("Sequence KD               : ALLOWED")
print("Feature KD                : ALLOWED AFTER PROJECTION AUDIT")
print("Direct logit KD           : PROHIBITED")
print("Test Set status           : SEALED")
print("Next                      : compute three-seed aggregate")


PHASE 03H CELL 3 STATUS
Status                    : PASSED
Finite metric runs        : 3 / 3 PASSED
Prediction counts         : 5000 / seed
Quality margin            : 3 / 3 VERIFIED
Overflow retries          : {2026: 8, 123: 8, 42: 10}
Sequence KD               : ALLOWED
Feature KD                : ALLOWED AFTER PROJECTION AUDIT
Direct logit KD           : PROHIBITED
Test Set status           : SEALED
Next                      : compute three-seed aggregate


## Cell 4 — Three-seed aggregate

### Công việc

- ghi một dòng cho mỗi seed;
- tính mean, sample SD (`n−1`) và CV;
- tổng hợp quality threshold/margin;
- không chọn lại checkpoint bằng Test.

### Đầu ra dự kiến

```text
CIDEr-D mean ± SD         : 1.326802 ± 0.003425
CIDEr-D CV                : 0.258124 %
All seeds qualified       : True
```


In [4]:
CELL_STARTED_AT = time.perf_counter()
metric_map = {
    "bleu_1": "BLEU-1",
    "bleu_4": "BLEU-4",
    "meteor": "METEOR",
    "rouge_l": "ROUGE-L",
    "cider_d": "CIDEr-D",
    "validation_ce": "validation_ce",
    "quality_threshold": None,
    "quality_margin": None,
}
per_run_rows = []
for run in sorted(run_records, key=lambda item: item["seed"]):
    marker = run["marker"]
    best = marker["best_result"]
    row = {
        "seed": run["seed"],
        "best_epoch": int(best["epoch"]),
        "bleu_1": float(best["BLEU-1"]),
        "bleu_4": float(best["BLEU-4"]),
        "meteor": float(best["METEOR"]),
        "rouge_l": float(best["ROUGE-L"]),
        "cider_d": float(best["CIDEr-D"]),
        "validation_ce": float(best["validation_ce"]),
        "quality_threshold": float(marker["quality_threshold"]),
        "quality_margin": float(marker["quality_margin"]),
        "overflow_retries": int(marker["overflow_retries"]),
        "best_checkpoint_sha256": marker["best_checkpoint"]["sha256"],
        "last_checkpoint_sha256": marker["last_checkpoint"]["sha256"],
        "quality_gate_passed": bool(marker["quality_gate_passed"]),
        "qualified_for_kd": bool(marker["qualified_for_kd"]),
        "test_set_status": marker["test_set_status"],
        "test_set_used": bool(marker["test_set_used"]),
    }
    per_run_rows.append(row)

PER_RUN_CSV = RESULT_DIR / "teacher_b_three_seed_per_run.csv"
with PER_RUN_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_run_rows[0]))
    writer.writeheader()
    writer.writerows(per_run_rows)

aggregate_metrics = {}
for field in metric_map:
    values = [float(row[field]) for row in per_run_rows]
    mean_value = statistics.mean(values)
    sd_value = statistics.stdev(values)
    aggregate_metrics[field] = {
        "mean": mean_value,
        "sample_sd": sd_value,
        "cv_percent": (
            100.0 * sd_value / abs(mean_value)
            if mean_value != 0.0
            else None
        ),
        "minimum": min(values),
        "maximum": max(values),
    }

AGGREGATE_JSON = RESULT_DIR / "teacher_b_three_seed_aggregate.json"
aggregate_payload = {
    "schema_version": "teacher_b_three_seed_aggregate_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "teacher_id": "B",
    "teacher": EXPECTED_MODEL,
    "revision": EXPECTED_REVISION,
    "seeds": [42, 123, 2026],
    "run_count": 3,
    "selection_split": "validation",
    "primary_metric": "CIDEr-D",
    "sample_standard_deviation": True,
    "metrics": aggregate_metrics,
    "all_quality_gates_passed": all(
        row["quality_gate_passed"] for row in per_run_rows
    ),
    "all_qualified_for_kd": all(
        row["qualified_for_kd"] for row in per_run_rows
    ),
    "kd_compatibility": KD_COMPATIBILITY,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(AGGREGATE_JSON, aggregate_payload)
assert math.isclose(
    aggregate_metrics["cider_d"]["mean"],
    1.3268017884706345,
    rel_tol=0.0,
    abs_tol=1e-12,
)
assert math.isclose(
    aggregate_metrics["cider_d"]["sample_sd"],
    0.0034247938873493066,
    rel_tol=0.0,
    abs_tol=1e-12,
)

write_json(
    AUDIT_DIR / "phase3h_cell04_aggregate.json",
    {
        "audit_version": "phase3h_cell04_aggregate_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "per_run_csv": str(PER_RUN_CSV),
        "aggregate_json": str(AGGREGATE_JSON),
        "cider_d": aggregate_metrics["cider_d"],
        "all_quality_gates_passed": True,
        "all_qualified_for_kd": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

cider = aggregate_metrics["cider_d"]
print("=" * 72)
print("PHASE 03H CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
for row in per_run_rows:
    print(
        f'Seed {row["seed"]:4d}                  : '
        f'CIDEr-D={row["cider_d"]:.9f} | '
        f'epoch={row["best_epoch"]} | '
        f'margin={row["quality_margin"]:.9f}'
    )
print(
    "CIDEr-D mean ± SD         :",
    f'{cider["mean"]:.9f} ± {cider["sample_sd"]:.9f}',
)
print("CIDEr-D CV                :", f'{cider["cv_percent"]:.6f} %')
print("All seeds qualified       : True")
print("Test Set status           : SEALED")
print("Per-run CSV               :", PER_RUN_CSV)
print("Aggregate JSON            :", AGGREGATE_JSON)
print("Next                      : build checkpoint registry and model card")


PHASE 03H CELL 4 STATUS
Status                    : PASSED
Seed   42                  : CIDEr-D=1.330165042 | epoch=8 | margin=0.331809603
Seed  123                  : CIDEr-D=1.323318605 | epoch=7 | margin=0.339937468
Seed 2026                  : CIDEr-D=1.326921718 | epoch=9 | margin=0.325981924
CIDEr-D mean ± SD         : 1.326801788 ± 0.003424794
CIDEr-D CV                : 0.258124 %
All seeds qualified       : True
Test Set status           : SEALED
Per-run CSV               : /kaggle/working//phase3_teacher_b_three_seed_aggregate/results/teacher_b_three_seed_per_run.csv
Aggregate JSON            : /kaggle/working//phase3_teacher_b_three_seed_aggregate/results/teacher_b_three_seed_aggregate.json
Next                      : build checkpoint registry and model card


## Cell 5 — Checkpoint registry và model card

### Công việc

- đăng ký best/last checkpoint của cả ba seed;
- ghi hash, đường dẫn relocatable và metric Validation;
- tạo model card ngắn gọn cho Teacher B;
- checkpoint vẫn không được deserialize.

### Đầu ra dự kiến

```text
Registry entries          : 3
Best checkpoints          : 3 / 3
Last checkpoints          : 3 / 3
Checkpoint payload read   : False
```


In [5]:
CELL_STARTED_AT = time.perf_counter()
registry_entries = []
for run in sorted(run_records, key=lambda item: item["seed"]):
    marker = run["marker"]
    best = marker["best_result"]
    registry_entries.append(
        {
            "teacher_id": "B",
            "seed": run["seed"],
            "model": EXPECTED_MODEL,
            "revision": EXPECTED_REVISION,
            "run_complete_marker": {
                "path": run["marker_path"],
                "sha256": run["marker_sha256"],
            },
            "best_checkpoint": {
                "path": str(
                    resolve_inside(
                        run["run_root"],
                        marker["best_checkpoint"]["relative_path"],
                    )
                ),
                "sha256": marker["best_checkpoint"]["sha256"],
                "selection_split": "validation",
                "selection_metric": "CIDEr-D",
                "best_epoch": int(best["epoch"]),
                "cider_d": float(best["CIDEr-D"]),
            },
            "last_checkpoint": {
                "path": str(
                    resolve_inside(
                        run["run_root"],
                        marker["last_checkpoint"]["relative_path"],
                    )
                ),
                "sha256": marker["last_checkpoint"]["sha256"],
                "completed_epochs": int(marker["completed_epochs"]),
                "optimizer_steps": int(marker["optimizer_steps"]),
            },
            "qualified_for_kd": True,
            "kd_compatibility": KD_COMPATIBILITY,
            "test_set_status": "sealed",
            "test_set_used": False,
        }
    )

REGISTRY_JSON = RESULT_DIR / "teacher_b_checkpoint_registry.json"
write_json(
    REGISTRY_JSON,
    {
        "schema_version": "teacher_b_checkpoint_registry_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "B",
        "entry_count": len(registry_entries),
        "entries": registry_entries,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

MODEL_CARD = MODEL_CARD_DIR / "teacher_b_model_card.md"
MODEL_CARD.write_text(
    "\n".join(
        [
            "# Teacher B — BLIP Base",
            "",
            f"- Source: `{EXPECTED_MODEL}`",
            f"- Revision: `{EXPECTED_REVISION}`",
            "- Dataset: MS COCO 2014 Train + Restval",
            "- Selection: Karpathy Validation only",
            "- Seeds: 42, 123, 2026",
            (
                "- CIDEr-D: "
                f'{aggregate_metrics["cider_d"]["mean"]:.9f} ± '
                f'{aggregate_metrics["cider_d"]["sample_sd"]:.9f} '
                "(sample SD)"
            ),
            (
                "- BLEU-4: "
                f'{aggregate_metrics["bleu_4"]["mean"]:.9f} ± '
                f'{aggregate_metrics["bleu_4"]["sample_sd"]:.9f}'
            ),
            "- Quality gate: 3/3 passed",
            "- Sequence KD: allowed",
            "- Feature KD: allowed only after projection audit",
            "- Direct logit KD: prohibited (vocabulary mismatch)",
            "- Held-out Test: sealed and unused",
            "",
        ]
    ),
    encoding="utf-8",
)

write_json(
    AUDIT_DIR / "phase3h_cell05_registry_model_card.json",
    {
        "audit_version": "phase3h_cell05_registry_model_card_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "registry_entries": len(registry_entries),
        "registry": str(REGISTRY_JSON),
        "model_card": str(MODEL_CARD),
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03H CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Registry entries          :", len(registry_entries))
print("Best checkpoints          : 3 / 3")
print("Last checkpoints          : 3 / 3")
print("Checkpoint payload read   : False")
print("Registry                  :", REGISTRY_JSON)
print("Model card                :", MODEL_CARD)
print("Test Set status           : SEALED")
print("Next                      : authorize Teacher C preflight")


PHASE 03H CELL 5 STATUS
Status                    : PASSED
Registry entries          : 3
Best checkpoints          : 3 / 3
Last checkpoints          : 3 / 3
Checkpoint payload read   : False
Registry                  : /kaggle/working//phase3_teacher_b_three_seed_aggregate/results/teacher_b_checkpoint_registry.json
Model card                : /kaggle/working//phase3_teacher_b_three_seed_aggregate/model_cards/teacher_b_model_card.md
Test Set status           : SEALED
Next                      : authorize Teacher C preflight


## Cell 6 — Teacher C entry contract

### Công việc

- ánh xạ Teacher C với S3 TinyViT-5M;
- chọn kiến trúc `SigLIP ViT-B/16 + shared caption decoder`;
- yêu cầu Phase 03I khóa exact offline repo/revision;
- chỉ cho phép logit KD khi decoder và vocabulary shared được strict-load;
- không tải Teacher C trong cell này.

### Đầu ra dự kiến

```text
Teacher C                : SigLIP ViT-B/16 + shared caption decoder
Matched Student          : S3 TinyViT-5M
Teacher C model loaded   : False
Phase 03I authorized     : True
```


In [6]:
CELL_STARTED_AT = time.perf_counter()
assert aggregate_payload["all_quality_gates_passed"] is True
assert aggregate_payload["all_qualified_for_kd"] is True

TEACHER_C_ENTRY = CONFIG_DIR / "teacher_c_entry_contract.json"
teacher_c_contract = {
    "contract_version": "phase3_teacher_c_entry_v1",
    "created_at_utc": now_utc(),
    "status": "authorized_for_preflight",
    "teacher_id": "C",
    "teacher_name": "SigLIP ViT-B/16 + shared caption decoder",
    "matched_student": {
        "student_id": "S3",
        "name": "TinyViT-5M",
    },
    "candidate": {
        "architecture": "SigLIP ViT-B/16",
        "suggested_repo_id": "google/siglip-base-patch16-224",
        "repo_and_revision_status": (
            "must be locked and verified offline in Phase 03I"
        ),
        "caption_decoder": (
            "reuse the locked shared caption decoder and vocabulary "
            "only after strict checkpoint compatibility audit"
        ),
    },
    "default_kd_channels": ["logit", "feature", "sequence"],
    "kd_conditions": {
        "logit": (
            "allowed only with identical shared decoder, vocabulary, "
            "token IDs and temperature contract"
        ),
        "feature": (
            "requires learned projection and explicit shape/layer audit"
        ),
        "sequence": (
            "requires deterministic generation and fixed decoding policy"
        ),
    },
    "phase03i_required_gates": [
        "exact offline source manifest and revision",
        "image preprocessing contract",
        "shared decoder strict load",
        "visual feature adapter shape contract",
        "forward/loss/backward finite",
        "one-batch overfit",
        "exact resume",
        "Validation-only metric backend",
        "Held-out Test sealed",
    ],
    "teacher_c_model_loaded": False,
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(TEACHER_C_ENTRY, teacher_c_contract)

TEACHER_C_AUTHORIZED = OUTPUT_ROOT / (
    "PHASE3_TEACHER_C_PREFLIGHT_AUTHORIZED.json"
)
write_json(
    TEACHER_C_AUTHORIZED,
    {
        "marker_version": "phase3_teacher_c_preflight_authorized_v1",
        "created_at_utc": now_utc(),
        "status": "authorized",
        "prerequisite": {
            "teacher_b_aggregate": str(AGGREGATE_JSON),
            "teacher_b_aggregate_sha256": sha256_file(AGGREGATE_JSON),
            "teacher_b_registry": str(REGISTRY_JSON),
            "teacher_b_registry_sha256": sha256_file(REGISTRY_JSON),
        },
        "entry_contract": {
            "path": str(TEACHER_C_ENTRY),
            "sha256": sha256_file(TEACHER_C_ENTRY),
        },
        "next_phase": "Phase 03I Teacher C offline preflight",
        "teacher_c_model_loaded": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(
    AUDIT_DIR / "phase3h_cell06_teacher_c_entry.json",
    {
        "audit_version": "phase3h_cell06_teacher_c_entry_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_c_entry": str(TEACHER_C_ENTRY),
        "teacher_c_authorized": str(TEACHER_C_AUTHORIZED),
        "teacher_c_model_loaded": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03H CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher C                 : SigLIP ViT-B/16 + shared decoder")
print("Matched Student           : S3 TinyViT-5M")
print("Teacher C model loaded    : False")
print("Training started          : False")
print("Phase 03I authorized      : True")
print("Test Set status           : SEALED")
print("Entry contract            :", TEACHER_C_ENTRY)
print("Authorization marker      :", TEACHER_C_AUTHORIZED)
print("Next                      : package Phase 03H")


PHASE 03H CELL 6 STATUS
Status                    : PASSED
Teacher C                 : SigLIP ViT-B/16 + shared decoder
Matched Student           : S3 TinyViT-5M
Teacher C model loaded    : False
Training started          : False
Phase 03I authorized      : True
Test Set status           : SEALED
Entry contract            : /kaggle/working//phase3_teacher_b_three_seed_aggregate/configs/teacher_c_entry_contract.json
Authorization marker      : /kaggle/working//phase3_teacher_b_three_seed_aggregate/PHASE3_TEACHER_C_PREFLIGHT_AUTHORIZED.json
Next                      : package Phase 03H


## Cell 7 — Package Phase 03H

### Công việc

- tạo manifest CSV/JSON và SHA256SUMS;
- tạo marker `PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json`;
- xác minh lại toàn bộ output nhỏ của Phase 03H;
- giữ checkpoint ở input, không sao chép checkpoint lớn sang output.

### Đầu ra dự kiến

```text
Status                    : COMPLETE
Teacher B seeds           : 3 / 3
Manifest verification     : ALL PASSED
Teacher C preflight       : AUTHORIZED
Test Set status           : SEALED
```


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase3h_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase3h_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase3h_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json"
)

artifact_paths = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV,
            MANIFEST_JSON,
            SHA256SUMS,
            COMPLETE_MARKER,
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = []
for path in artifact_paths:
    manifest_rows.append(
        {
            "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase3h_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

for row in manifest_rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase3_teacher_b_three_seed_aggregate_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "teacher_id": "B",
        "model": EXPECTED_MODEL,
        "revision": EXPECTED_REVISION,
        "seeds": [42, 123, 2026],
        "run_count": 3,
        "aggregate": {
            "relative_path": AGGREGATE_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AGGREGATE_JSON),
        },
        "checkpoint_registry": {
            "relative_path": REGISTRY_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(REGISTRY_JSON),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "all_quality_gates_passed": True,
        "all_qualified_for_kd": True,
        "teacher_c_preflight_authorized": True,
        "teacher_c_authorization_marker": {
            "relative_path": TEACHER_C_AUTHORIZED.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(TEACHER_C_AUTHORIZED),
        },
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 03H CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher B seeds           : 3 / 3")
print("Quality gates             : 3 / 3 PASSED")
print("Qualified for KD          : 3 / 3")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Teacher C preflight       : AUTHORIZED")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Complete marker           :", COMPLETE_MARKER)
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Next                      : Save Version, then run Phase 03I")


PHASE 03H CELL 7 STATUS
Status                    : COMPLETE
Teacher B seeds           : 3 / 3
Quality gates             : 3 / 3 PASSED
Qualified for KD          : 3 / 3
Manifest rows             : 13
Manifest verification     : ALL PASSED
Teacher C preflight       : AUTHORIZED
Checkpoint payload read   : False
Test Set status           : SEALED
Test Set used             : False
Complete marker           : /kaggle/working//phase3_teacher_b_three_seed_aggregate/PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json
Elapsed                   : 0.02 giây
Next                      : Save Version, then run Phase 03I
